# Audit di Qualità e Integrità dei Dataset Puliti

Questo notebook esegue un'analisi approfondita di qualità sui due dataset consolidati e normalizzati:
1. **`Fumetti_clean.csv`** (Catalogo albi DC & Marvel)
2. **`Personaggi_clean.csv`** (Anagrafica personaggi DC & Marvel)

L'obiettivo è verificare la completezza dei dati, l'assenza di anomalie bloccanti e l'integrità referenziale in vista della modellazione del grafo su Neo4j.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'Scripts' else Path.cwd()
PULITI_DIR = BASE_DIR / 'Dataset' / 'Puliti'

# Caricamento dataset
fumetti = pd.read_csv(PULITI_DIR / 'Fumetti_clean.csv')
personaggi = pd.read_csv(PULITI_DIR / 'Personaggi_clean.csv')

print(f"Fumetti caricati:    {fumetti.shape[0]:,} righe x {fumetti.shape[1]} colonne")
print(f"Personaggi caricati: {personaggi.shape[0]:,} righe x {personaggi.shape[1]} colonne")

Fumetti caricati:    60,919 righe x 9 colonne
Personaggi caricati: 23,269 righe x 14 colonne


## 1. Analisi di Qualità: `Fumetti_clean.csv`

In [2]:
print("=== 1.1 VERIFICA CHIAVE PRIMARIA (issue_id) ===")
dup_ids = fumetti['issue_id'].duplicated().sum()
null_ids = fumetti['issue_id'].isnull().sum()
print(f"Duplicati su issue_id: {dup_ids} (OK se 0)")
print(f"Valori nulli su issue_id: {null_ids} (OK se 0)")

print("\n=== 1.2 COMPLETEZZA DEI DATI (Valori NaN e 'Unknown') ===")
quality_fumetti = pd.DataFrame({
    'Tipo Dato': fumetti.dtypes,
    'Valori Nulli (NaN)': fumetti.isnull().sum(),
    'Valori Unknown': (fumetti == 'Unknown').sum(),
    '% Unknown/Mancanti': (((fumetti == 'Unknown').sum() + fumetti.isnull().sum()) / len(fumetti) * 100).round(2)
})
quality_fumetti

=== 1.1 VERIFICA CHIAVE PRIMARIA (issue_id) ===
Duplicati su issue_id: 0 (OK se 0)
Valori nulli su issue_id: 0 (OK se 0)

=== 1.2 COMPLETEZZA DEI DATI (Valori NaN e 'Unknown') ===


,Tipo Dato,Valori Nulli (NaN),Valori Unknown,% Unknown/Mancanti
issue_id,object,0,0,0.00
publisher,object,0,0,0.00
series_title,object,0,0,0.00
issue_title,object,0,0,0.00
release_year,object,0,593,0.97
comic_type,object,0,1566,2.57
writer,object,0,8814,14.47
penciler,object,0,9818,16.12
cover_artist,object,0,23115,37.94


In [3]:
print("=== 1.3 DISTRIBUZIONE PER EDITORE ===")
print(fumetti['publisher'].value_counts())

print("\n=== 1.4 ANALISI TEMPORALE (release_year) ===")
years_num = pd.to_numeric(fumetti['release_year'], errors='coerce')
valid_years = years_num.dropna()
print(f"Anni validi estratti: {len(valid_years):,} / {len(fumetti):,} ({len(valid_years)/len(fumetti)*100:.1f}%)")
print(f"Anno minimo: {int(valid_years.min())} | Anno massimo: {int(valid_years.max())}")

# Dettaglio anni anomali o placeholder Marvel
future_years = fumetti[years_num > 2026]
print(f"Record con anno futuro (>2026, placeholder Marvel API come 2029): {len(future_years)}")

print("\n=== 1.5 TIPOLOGIE DI FORMATO (comic_type) ===")
print(fumetti['comic_type'].value_counts().head(10))

=== 1.3 DISTRIBUZIONE PER EDITORE ===
publisher
Marvel    34332
DC        26587
Name: count, dtype: int64

=== 1.4 ANALISI TEMPORALE (release_year) ===
Anni validi estratti: 60,326 / 60,919 (99.0%)
Anno minimo: 1935 | Anno massimo: 2029
Record con anno futuro (>2026, placeholder Marvel API come 2029): 25

=== 1.5 TIPOLOGIE DI FORMATO (comic_type) ===
comic_type
Comic              31780
Category           25015
Stand-Alone         1572
Unknown             1566
Digital Comic        345
Infinite Comic       265
Trade Paperback      215
Hardcover             73
Magazine              61
Digest                 8
Name: count, dtype: int64


## 2. Analisi di Qualità: `Personaggi_clean.csv`

In [4]:
print("=== 2.1 VERIFICA CHIAVE PRIMARIA (character_id) ===")
dup_char_ids = personaggi['character_id'].duplicated().sum()
null_char_ids = personaggi['character_id'].isnull().sum()
print(f"Duplicati su character_id: {dup_char_ids} (OK se 0)")
print(f"Valori nulli su character_id: {null_char_ids} (OK se 0)")

print("\n=== 2.2 COMPLETEZZA DEI DATI PERSONAGGI ===")
quality_personaggi = pd.DataFrame({
    'Tipo Dato': personaggi.dtypes,
    'Valori Nulli (NaN)': personaggi.isnull().sum(),
    '% Mancanti': (personaggi.isnull().sum() / len(personaggi) * 100).round(2)
})
quality_personaggi

=== 2.1 VERIFICA CHIAVE PRIMARIA (character_id) ===
Duplicati su character_id: 0 (OK se 0)
Valori nulli su character_id: 0 (OK se 0)

=== 2.2 COMPLETEZZA DEI DATI PERSONAGGI ===


,Tipo Dato,Valori Nulli (NaN),% Mancanti
character_id,object,0,0.00
publisher,object,0,0.00
original_page_id,int64,0,0.00
name,object,0,0.00
url_slug,object,0,0.00
identity_status,object,0,0.00
alignment,object,0,0.00
eye_color,object,0,0.00
hair_color,object,0,0.00
gender,object,0,0.00


In [5]:
print("=== 2.3 DISTRIBUZIONE PER EDITORE ===")
print(personaggi['publisher'].value_counts())

print("\n=== 2.4 ALLINEAMENTO MORALE (alignment) ===")
print(personaggi['alignment'].value_counts())

print("\n=== 2.5 STATO IN VITA (status) ===")
print(personaggi['status'].value_counts())

print("\n=== 2.6 TOP 5 PERSONAGGI PIÙ PRESENTI PER EDITORE ===")
for pub in ['DC', 'Marvel']:
    top = personaggi[personaggi['publisher'] == pub].nlargest(5, 'appearances_count')
    print(f"\nTop 5 {pub}:")
    for _, r in top.iterrows():
        print(f"  - {r['name']} ({int(r['appearances_count'])} presenze, 1ª apparizione: {r['first_appearance_year']})")

=== 2.3 DISTRIBUZIONE PER EDITORE ===
publisher
Marvel    16373
DC         6896
Name: count, dtype: int64

=== 2.4 ALLINEAMENTO MORALE (alignment) ===
alignment
Bad        9615
Good       7471
Unknown    3410
Neutral    2773
Name: count, dtype: int64

=== 2.5 STATO IN VITA (status) ===
status
Living      17810
Deceased     5459
Name: count, dtype: int64

=== 2.6 TOP 5 PERSONAGGI PIÙ PRESENTI PER EDITORE ===

Top 5 DC:
  - Batman (Bruce Wayne) (3093 presenze, 1ª apparizione: 1939.0)
  - Superman (Clark Kent) (2496 presenze, 1ª apparizione: 1986.0)
  - Green Lantern (Hal Jordan) (1565 presenze, 1ª apparizione: 1959.0)
  - James Gordon (New Earth) (1316 presenze, 1ª apparizione: 1987.0)
  - Richard Grayson (New Earth) (1237 presenze, 1ª apparizione: 1940.0)

Top 5 Marvel:
  - Spider-Man (Peter Parker) (4043 presenze, 1ª apparizione: 1962.0)
  - Captain America (Steven Rogers) (3360 presenze, 1ª apparizione: 1941.0)
  - Wolverine (James "Logan" Howlett) (3061 presenze, 1ª apparizione: 1974

## 3. Riepilogo e Idoneità per il Grafo Neo4j

| Verifica | Fumetti (`Fumetti_clean.csv`) | Personaggi (`Personaggi_clean.csv`) | Esito |
| :--- | :--- | :--- | :--- |
| **Chiave Primaria Univoca** | `issue_id` univoco al 100% (0 duplicati) | `character_id` univoco al 100% (0 duplicati) | Conforme |
| **Valori Nulli (NaN)** | 0% (tutti gestiti come `'Unknown'`) | Presenti solo su `appearances` (6.2%) e `year` (3.8%) | Conforme |
| **Identificativo Editore** | `'DC'` e `'Marvel'` normalizzati | `'DC'` e `'Marvel'` normalizzati | Conforme |
| **Consistenza Temporale** | 99% degli albi ha un anno valido (1935-2024) | 96.2% dei personaggi ha anno di debutto | Conforme |
| **Pronto per Neo4j** | Sì (Nodi `:Issue`, `:Series`, `:Creator`) | Sì (Nodi `:Character`, Proprietà Morali/Bio) | Conforme |